# Credit Risk
## От прогноза просрочки к анализу кредитных решений

**Give Me Some Credit · SQL · Python · ML · риск-аналитика**

Цель: сравнить модели риска серьёзной просрочки и показать, как выбор порога меняет долю условных одобрений и пропуск проблемных заёмщиков.

**Статус:** обновлённый эксперимент ещё не выполнен. Все вычисляемые outputs появятся после пользовательского запуска. Исторические результаты отдельно помечены.

[Репозиторий](https://github.com/Arti-D/credit-risk-project) · [Датасет](https://www.kaggle.com/competitions/GiveMeSomeCredit/data)

## 1. Среда и данные

Рекомендуемый запуск — JupyterLab на своём компьютере. Можно использовать Colab с локальной средой. Обычная размещённая среда Colab использует серверы Google.

Скачайте ZIP ветки проекта и распакуйте его. Установите зависимости из requirements.txt. Положите **cs-training.csv** в **data/raw/**. Не используйте cs-test.csv: в нём нет целевой переменной.

Следующая ячейка только настраивает пути; при необходимости укажите PROJECT_ROOT вручную.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
# Если notebook открыт не из папки проекта, укажите путь:
# PROJECT_ROOT = Path(r"C:/path/to/credit-risk-project")
if not (PROJECT_ROOT / "src").is_dir() and (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").is_dir():
    raise FileNotFoundError("Укажите PROJECT_ROOT — папку, содержащую src/ и requirements.txt")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project:", PROJECT_ROOT)

**Установка зависимостей (один раз).** Убирайте # только если зависимости ещё не установлены. Установка ничего не обучает.

In [ ]:
# %pip install -r {PROJECT_ROOT / "requirements.txt"}

In [ ]:
import json
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
from preprocess import ROOT, TARGET, load_frame, load_splits

plt.rcParams.update({"figure.figsize": (9, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
pd.set_option("display.max_columns", 20)

## 2. Загрузка и качество данных

**Следующая ячейка читает весь CSV и создаёт SQLite-базу.** Это обработка данных, не обучение. Запускайте её сами после загрузки CSV.

Цель SeriousDlqin2yrs: серьёзная просрочка (90+ дней) в двухлетнем горизонте. Идентификатор не используется как признак.

Проверяем пропуски, необычные значения и баланс классов. Графики ниже носят описательный характер; решения о моделях принимаются на train/validation, не на test.

In [ ]:
from load_data import main as load_data
load_data()
df = load_frame()
display(pd.DataFrame({"rows": [len(df)], "features": [len(df.columns)-2],
                      "event_rate": [df[TARGET].mean()]}))
display(df.head())

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]
if len(missing):
    ax = (missing * 100).sort_values().plot.barh(color="#247e91")
    ax.set(title="Пропуски в исходных данных", xlabel="Доля, %")
    plt.tight_layout(); plt.show()
display(df.describe().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
df[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], color=["#247e91", "#df8755"], rot=0)
axes[0].set(title="Баланс целевого события", xlabel="0 — нет, 1 — есть", ylabel="Клиенты")
df["age"].hist(bins=35, ax=axes[1], color="#247e91", edgecolor="white")
axes[1].set(title="Возраст: исходные значения", xlabel="Возраст", ylabel="Клиенты")
plt.tight_layout(); plt.show()

### SQL: риск по сегментам

Сравниваем наблюдаемую частоту события, а не только средние значения признаков. Связи описательные и не доказывают причинность.

In [ ]:
from eda import run_sql_queries
with sqlite3.connect(ROOT / "data/processed/credit.db") as conn:
    run_sql_queries(conn)

age_band = pd.cut(df["age"], [18, 30, 45, 60, float("inf")],
                  labels=["18–29", "30–44", "45–59", "60+"], right=False)
segment = df.groupby(age_band, observed=True)[TARGET].agg(["size", "mean"])
segment.columns = ["clients", "observed_event_rate"]
display(segment)
ax = (segment["observed_event_rate"] * 100).plot.bar(color="#247e91", rot=0)
ax.set(title="Наблюдаемая просрочка по возрастным группам", xlabel="Возрастная группа", ylabel="Частота, %")
plt.tight_layout(); plt.show()

## 3. Дизайн эксперимента

- **Train 60%:** обучение преобразований и моделей.
- **Validation 20%:** выбор модели по ROC-AUC, выбор порога для заранее заданного recall 80%.
- **Test 20%:** финальная оценка после фиксации модели и порога.

Медианы и 99-й перцентиль clipping обучаются только на train. Пропуски дохода/иждивенцев и необычные значения имеют индикаторы. Счётчики просрочек ≥90 трактуются как неизвестные — это проверяемое допущение.

Случайное разделение не заменяет временную валидацию. В Optuna вся подготовка данных находится внутри CV.

In [ ]:
X_train, X_valid, X_test, y_train, y_valid, y_test = load_splits()
display(pd.DataFrame([{"split": name, "rows": len(y), "event_rate": y.mean()}
    for name, y in [("train", y_train), ("validation", y_valid), ("test", y_test)]]))

## 4. Обучение — запускает пользователь

**Эта ячейка обучает три модели и использует CPU.** По умолчанию выключена. Поменяйте RUN_TRAINING на True, когда готовы.

Логистическая регрессия — baseline; Random Forest и XGBoost проверяют выигрыш нелинейных зависимостей. Веса классов заданы как исходная конфигурация, а не доказанный оптимум. Optuna сегодня необязательна.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from train import main as train_models
    train_models()
else:
    print("Обучение не запущено. Для запуска установите RUN_TRAINING = True.")

## 5. Финальная оценка — запускает пользователь

**ROC-AUC** оценивает ранжирование. **Average Precision (AP)** дополняет его при дисбалансе; это не трапецеидальная PR-AUC. Точечные метрики считаются на самом test, bootstrap даёт интервалы.

Модель и порог фиксируются на validation. Test используется один раз для отчёта, без дальнейшего подбора по нему. 200 bootstrap-повторов — быстрый предварительный вариант; 1000 дольше.

Ячейка не обучает модели, но выполняет предсказания и bootstrap, поэтому тоже выключена по умолчанию.

In [ ]:
RUN_EVALUATION = False
if RUN_EVALUATION:
    from evaluate import main as evaluate_models
    evaluate_models(n_bootstrap=200, target_recall=.8)
else:
    print("Оценка не запущена. После обучения установите RUN_EVALUATION = True.")

In [ ]:
reports = ROOT / "reports"
for name in ["validation_metrics.csv", "test_metrics.csv", "threshold_scenarios.csv", "risk_bands.csv"]:
    file = reports / name
    if file.exists():
        print(name); display(pd.read_csv(file))
    else:
        print(name, "— ожидает вашего запуска")
for name in ["policy.json", "paired_comparison.json"]:
    file = reports / name
    if file.exists():
        print(name); display(json.loads(file.read_text(encoding="utf-8")))

In [ ]:
for name in ["discrimination.png", "calibration.png", "risk_bands.png"]:
    file = reports / "figures" / name
    if file.exists():
        display(Image(filename=str(file)))

## 6. Как читать результат для риск-решений

**Approval rate:** доля заявок со score ниже порога. **Observed bad rate approved:** наблюдаемая частота серьёзной просрочки среди условно одобренных. **Recall:** доля проблемных клиентов, которых правило отклоняет.

Низкий порог обычно снижает одобрения и повышает выявление проблемных клиентов. Порог 80% recall — учебный ориентир на validation, а не гарантированный test-результат или оптимальная политика банка.

Сценарии одобрения — ретроспективная симуляция. Нет данных о реальных отказах, EAD, LGD, доходности и затратах, поэтому расчёт прибыли не выполняется. Взвешенные модели требуют проверки калибровки; score нельзя автоматически называть откалиброванной PD.

**После запуска заполнить выводы фактическими цифрами:**
1. Какая модель выбрана на validation? Каковы test ROC-AUC и AP?
2. Как меняются одобрения и риск среди одобренных при разных порогах?
3. Содержит ли парный интервал разницы ROC-AUC с baseline ноль?
4. Где модель ошибается и какие проверки нужны перед применением?

## Архив: результат предыдущей версии

Прежняя версия показывала ROC-AUC ≈0.872 и AP ≈0.404 для tuned XGBoost. **Эти значения не являются результатом кода выше:** preprocessing вычислялся до split, а точечные метрики были средними bootstrap-оценками.

Архив нужен для прозрачности истории проекта. После нового запуска используем reports/test_metrics.csv и не подменяем его прежними цифрами.

In [ ]:
historical = ROOT / "models/final_results.csv"
if historical.exists():
    print("АРХИВ прежнего эксперимента, не новый результат")
    display(pd.read_csv(historical))

## Воспроизводимость и ограничения

Seed 42, стратифицированный split, версии библиотек и SHA-256 CSV сохраняются в models/current/run.json. Модели и индивидуальные предсказания не публикуются.

Ограничения: нет временной валидации, отдельной калибровки и проверки справедливости; обработка аномалий требует сравнения альтернатив внутри CV. Наблюдательные данные не описывают эффект изменения кредитной политики.

Для передачи результатов сохраните выполненный notebook и пришлите reports/test_metrics.csv, reports/policy.json и reports/threshold_scenarios.csv.